
# Unemployment Analysis in India: Covid‑19 Impact Study

## Project Goal
This project investigates unemployment trends in India and evaluates how Covid‑19 affected employment conditions.

### Objectives
- Investigate unemployment changes during Covid‑19
- Identify seasonal and monthly patterns
- Compare states and regions
- Generate insights for policy recommendations

### Dataset
1. Unemployment in India dataset  
2. Covid period unemployment dataset (optional merge)


## 1. Import Libraries
Load libraries for analysis, cleaning and visualization.

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import calendar
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

plt.rcParams["figure.figsize"] = (12,6)

sns.set_style("whitegrid")
sns.set_palette("Set2")

## 2. Load Dataset
Read raw unemployment dataset.

In [116]:
data_path = next(Path.cwd().glob('**/Unemployment in India.csv'), None)
if data_path is None:
    raise FileNotFoundError("Could not find 'Unemployment in India.csv' in the current folder tree.")
df = pd.read_csv(data_path)

## 3. Dataset Overview
Check shape and dimensions.

In [117]:
print("Dataset 1 Shape:", df.shape)

Dataset 1 Shape: (768, 7)


## 4. Column Inspection
Review column names before cleaning.

In [118]:
print("\nDataset Columns")
print(df.columns)


Dataset Columns
Index(['Region', ' Date', ' Frequency', ' Estimated Unemployment Rate (%)',
       ' Estimated Employed', ' Estimated Labour Participation Rate (%)',
       'Area'],
      dtype='str')


## 5. Initial Preview
Display first records.

In [119]:
df.head()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


In [120]:
df.tail()

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
763,NaN,NaN,NaN,NaN,NaN,NaN,NaN
764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
765,NaN,NaN,NaN,NaN,NaN,NaN,NaN
766,NaN,NaN,NaN,NaN,NaN,NaN,NaN
767,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Data Information
Check datatypes and missing values.

In [121]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Region                                    740 non-null    str    
 1    Date                                     740 non-null    str    
 2    Frequency                                740 non-null    str    
 3    Estimated Unemployment Rate (%)          740 non-null    float64
 4    Estimated Employed                       740 non-null    float64
 5    Estimated Labour Participation Rate (%)  740 non-null    float64
 6   Area                                      740 non-null    str    
dtypes: float64(3), str(4)
memory usage: 42.1 KB


## 7. Statistical Summary

In [122]:
df.describe()

,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%)
count,740.000000,7.400000e+02,740.000000
mean,11.787946,7.204460e+06,42.630122
std,10.721298,8.087988e+06,8.111094
min,0.000000,4.942000e+04,13.330000
25%,4.657500,1.190404e+06,38.062500
50%,8.350000,4.744178e+06,41.160000
75%,15.887500,1.127549e+07,45.505000
max,76.740000,4.577751e+07,72.570000


In [123]:
df.Region.value_counts()

Region
Andhra Pradesh      28
Bihar               28
Chhattisgarh        28
Delhi               28
Gujarat             28
Haryana             28
Himachal Pradesh    28
Jharkhand           28
Karnataka           28
Kerala              28
Madhya Pradesh      28
Maharashtra         28
Odisha              28
Punjab              28
Rajasthan           28
Tamil Nadu          28
Telangana           28
Tripura             28
Uttar Pradesh       28
West Bengal         28
Meghalaya           27
Uttarakhand         27
Assam               26
Puducherry          26
Goa                 24
Jammu & Kashmir     21
Sikkim              17
Chandigarh          12
Name: count, dtype: int64

## 8. Rename columns

In [124]:
df.columns = ['state','date','frequency','estimated unemployment rate','estimated employed','estimated labour participation rate','area']
df.head()

,state,date,frequency,estimated unemployment rate,estimated employed,estimated labour participation rate,area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


## 9. Data Cleaning and Missing Value Analysis
Remove spaces, and fix formats.

In [125]:
df.columns = df.columns.str.strip()

In [126]:
print(df.isnull().sum())

state                                  28
date                                   28
frequency                              28
estimated unemployment rate            28
estimated employed                     28
estimated labour participation rate    28
area                                   28
dtype: int64


In [127]:
df = df.dropna()
print("Null values remove")

Null values remove


In [128]:
print(df.isnull().sum())

state                                  0
date                                   0
frequency                              0
estimated unemployment rate            0
estimated employed                     0
estimated labour participation rate    0
area                                   0
dtype: int64


In [129]:
print(
    "Dataset duplicates:",
    df.duplicated().sum()
)

Dataset duplicates: 0


## 10. Feature Engineering
Create month and time features.

In [130]:
df['date'] = pd.to_datetime(df['date'],dayfirst = True)
df['month_int'] = df['date'].dt.month
df['month'] = df['month_int'].apply(lambda x: calendar.month_abbr[int(x)] if pd.notnull(x) else np.nan)
df.head()

,state,date,frequency,estimated unemployment rate,estimated employed,estimated labour participation rate,area,month_int,month
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139.0,43.24,Rural,5,May
1,Andhra Pradesh,2019-06-30,Monthly,3.05,11755881.0,42.05,Rural,6,Jun
2,Andhra Pradesh,2019-07-31,Monthly,3.75,12086707.0,43.50,Rural,7,Jul
3,Andhra Pradesh,2019-08-31,Monthly,3.32,12285693.0,43.97,Rural,8,Aug
4,Andhra Pradesh,2019-09-30,Monthly,5.17,12256762.0,44.68,Rural,9,Sep


## 11. Exploratory Data Analysis

In [131]:
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

monthly_avg = (
    df.groupby('date')['estimated unemployment rate']
    .mean()
    .reset_index()
)

fig = px.line(
    monthly_avg,
    x='date',
    y='estimated unemployment rate',
    title='India Unemployment Trend Over Time',
    markers=True
)

fig.add_vrect(
    x0='2020-03-01',
    x1='2020-06-30',
    annotation_text='Covid Lockdown',
    fillcolor='red',
    opacity=0.2
)

fig.show()

In [132]:
covid = df[df['date'] >= '2020-03-01']

state_avg = (
    covid.groupby('state')['estimated unemployment rate']
    .mean()
    .reset_index()
)

fig = px.bar(
    state_avg.sort_values(
        'estimated unemployment rate',
        ascending=False
    ),
    x='state',
    y='estimated unemployment rate',
    color='estimated unemployment rate',
    text_auto='.1f',
    title='Average Unemployment During Covid'
)

fig.show()



## 12. Covid Impact Analysis

In [133]:
before = df[df['date'] < '2020-03-01']
during = df[df['date'] >= '2020-03-01']

before_avg = before[
    'estimated unemployment rate'
].mean()

during_avg = during[
    'estimated unemployment rate'
].mean()

compare = pd.DataFrame({
    'Period':[
        'Before Covid',
        'During Covid'
    ],
    'Rate':[
        before_avg,
        during_avg
    ]
})

# percentage increase
increase = (
    (during_avg-before_avg)
    / before_avg
) * 100

fig = px.bar(
    compare,
    x='Period',
    y='Rate',
    color='Period',
    text='Rate',
    title='Unemployment Rate: Before vs During Covid'
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig.add_annotation(
    x='During Covid',
    y=during_avg,
    text=f'↑ {increase:.1f}% increase',
    showarrow=True,
    arrowhead=3
)

fig.update_layout(
    xaxis_title='Period',
    yaxis_title='Average Unemployment Rate (%)',
    template='plotly_white',
    height=600,
    showlegend=False
)

fig.show()

In [134]:
# Convert date column
df['date'] = pd.to_datetime(df['date'])

# Create month column again
df['month'] = df['date'].dt.month_name()

# Check result
print(df[['date','month']].head())

season = (
    df.groupby('month')
    ['estimated unemployment rate']
    .mean()
    .reset_index()
)

month_order = [
    'January','February','March',
    'April','May','June',
    'July','August','September',
    'October','November','December'
]

season['month'] = pd.Categorical(
    season['month'],
    categories=month_order,
    ordered=True
)

season = season.sort_values('month')

fig = px.line(
    season,
    x='month',
    y='estimated unemployment rate',
    markers=True,
    title='Seasonal Trend and Covid Impact on Unemployment'
)

fig.update_traces(
    line=dict(width=4),
    marker=dict(size=10),
    fill='tozeroy'
)

fig.show()

        date      month
0 2019-05-31        May
1 2019-06-30       June
2 2019-07-31       July
3 2019-08-31     August
4 2019-09-30  September


## 13. Seasonal Trend Analysis

In [135]:
area_data = (
    df.groupby(['date','area'])
    ['estimated unemployment rate']
    .mean()
    .reset_index()
)

fig = px.line(
    area_data,
    x='date',
    y='estimated unemployment rate',
    color='area',
    markers=True,
    title='Rural vs Urban Unemployment During Covid'
)

# Improve lines
fig.update_traces(
    line=dict(width=4),
    marker=dict(size=8)
)

# Highlight lockdown period
fig.add_vrect(
    x0='2020-03-01',
    x1='2020-06-01',
    fillcolor='red',
    opacity=0.15,
    annotation_text='Covid Lockdown',
    annotation_position='top left'
)

# Find peak unemployment
peak = area_data.loc[
    area_data[
        'estimated unemployment rate'
    ].idxmax()
]

fig.add_annotation(
    x=peak['date'],
    y=peak[
        'estimated unemployment rate'
    ],
    text='Peak Unemployment',
    showarrow=True,
    arrowhead=2
)

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    height=650,
    xaxis_title='date',
    yaxis_title='Average Unemployment Rate (%)',
    legend_title='Area Type',
    title_x=0.5
)

fig.show()

In [136]:
top_states = (
    covid.groupby('state')
    ['estimated unemployment rate']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_states,
    x='estimated unemployment rate',
    y='state',
    orientation='h',
    color='estimated unemployment rate',
    text='estimated unemployment rate',
    title='Top 10 States Most Affected by Covid',
    color_continuous_scale='Reds'
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig.update_layout(
    yaxis=dict(
        categoryorder='total ascending'
    ),
    xaxis_title='average unemployment rate',
    yaxis_title='state',
    template='plotly_white',
    hovermode='y unified',
    height=650,
    coloraxis_colorbar_title='Rate %'
)

fig.show()

## 14. State Comparison

In [137]:
fig = px.scatter(
    df,
    x='estimated employed',
    y='estimated unemployment rate',
    color='area',
    hover_data=['state'],
    title='Employment vs Unemployment'
)

fig.show()

In [138]:
fig = px.scatter(
    df,
    x='estimated labour participation rate',
    y='estimated unemployment rate',
    color='state',
    title='Labour Participation vs Unemployment'
)

fig.show()

## 15. Final Insights
Summarize findings and recommendations.

In [139]:
shock = (
    df.groupby('date')
    ['estimated unemployment rate']
    .mean()
    .reset_index()
)

fig = px.area(
    shock,
    x='date',
    y='estimated unemployment rate',
    title='Covid Shock Timeline'
)

fig.show()

In [140]:
fig = px.sunburst(
    state_avg,
    path=['state'],
    values='estimated unemployment rate',
    title='Unemployment rate by state',
    height=600
)
fig.show()


# Conclusion

Key observations:
- Covid period showed significant unemployment increase
- Some states were affected more severely
- Seasonal variation exists across months
- Results can support labour and economic planning
